# Aralin 18: Pag-secure ng mga AI Agent gamit ang Cryptographic Receipts

## Hands-on Notebook

Ang notebook na ito ay naglalakad sa apat na ehersisyo:

1. **Pirmahan ang iyong unang resibo** para sa isang pagtawag ng agent tool at patunayan ito.
2. **Pekein ang resibo** at panoorin ang kabiguan ng beripikasyon.
3. **Gumawa ng tatlong-resibo na chain** at kumpirmahin ang integridad ng chain.
4. **Balutin ang pagtawag ng Microsoft Agent Framework tool** para bawat aksyon ay maglabas ng resibo.

Lahat ng cryptographic primitives ay inangkat mula sa mga mahusay at maayos na library (`pynacl` para sa Ed25519, `jcs` para sa RFC 8785 canonical JSON, `hashlib` mula sa Python standard library para sa SHA-256). Ang lohika ng resibo mismo ay plain Python na maaari mong basahin at baguhin.

Patakbuhin ang mga cells nang sunod-sunod. Ang bawat seksyon ay maikli at self-contained.


## Setup

I-install ang dalawang dependencies. Pareho silang may maluwag na mga lisensya (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Mga Tulong na Utility

Ang dalawang tulong na ito ay humahawak sa base64url encoding (nang walang padding) at SHA-256 hashing ng mga arbitrary na bagay. Pinananatili nila ang natitirang bahagi ng notebook na nakatuon sa lohika ng resibo mismo.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Seksyon 1: Lagdaan ang iyong unang resibo

Isipin na ang aming ahente para sa **Contoso Travel** ay kakakita lang ng mga flight mula Sydney papuntang Los Angeles para sa isang customer. Gusto nating irekord ang tawag sa tool na ito bilang isang nilagdaang resibo upang ang isang auditor sa hinaharap ay makapagpatunay nito nang hindi na kailangang magtiwala sa amin.

### Hakbang 1.1: Lumikha ng isang susi para sa paglagda

Sa produksyon, ang susi ng ahente para sa paglagda ay karaniwang matatagpuan sa isang hardware security module (HSM), Azure Key Vault, o isang katulad na protektadong imbakan. Para sa araling ito, gagawa tayo ng bagong susi sa memorya.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Hakbang 1.2: Buoin ang payload ng resibo

Ang payload ay naglalaman ng lahat ng nais nating patunayan ng resibo: sino ang kumilos, anong kasangkapan, gamit ang anong mga argumento, ano ang bumalik, sa ilalim ng anong patakaran, at kailan. I-hash namin ang mga argumento at resulta sa halip na isama ang mga ito nang direkta upang hindi magtagas ng sensitibong nilalaman ang resibo.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Hakbang 1.3: Pirmahan at tipunin ang resibo

Tatlong hakbang:

1. I-canonicalize ang payload gamit ang JCS upang ang dalawang implementasyon na gumagawa ng parehong lohikal na resibo ay makagawa ng byte-identical bytes.
2. I-hash ang canonical bytes gamit ang SHA-256.
3. Pirmahan ang hash gamit ang Ed25519 private key.

Ang pirma ay ikinakabit pagkatapos sa orihinal na payload upang mabuo ang pinal na resibo.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Hakbang 1.4: Suriin ang resibo

Ang pag-verify ay binaliktad na proseso. Inaalis namin ang pirma, muling kinakalkula ang canonical na hash, at sinusuri ang pirma laban sa pampublikong susi sa resibo.

Ang isang auditor na gumagawa ng pagsusuring ito ay hindi nangangailangan ng anuman mula sa amin maliban sa mismong resibo. Walang serbisyong kailangang tawagan, walang direktoryo ng susi na kailangang tanungin, walang kinakailangang tiwala.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Dapat mong makita ang `Receipt is valid: True`. Ang ahente ay nakabuo ng kanyang unang cryptographically signed audit record.


## Seksyon 2: Pagsabotahe sa resibo

Ang buong punto ng mga resibo ay ang mga ito ay madaling makita kung may kinialaman. Patunayan natin ito.

Babaguhin natin ang eksaktong isang karakter ng resibo at panoorin ang pagkabigo ng beripikasyon.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Ano ang nangyari?

Nang binago natin ang `policy_id`, nagbago ang mga canonical bytes. Nagbago ang SHA-256 hash ng mga bytes na iyon. Ang lagda (na ginawa batay sa orihinal na hash) ay hindi na tumutugma sa bagong hash. Tama ang pagbalik ng Verification bilang `False`.

Wala nang paraan para baguhin ang kahit anong field ng resibo at mapatotohanan pa rin ito, maliban na lang kung hawak ng attacker ang private key. Hangga't ang private key ay nasa key vault at ang public key ay naka-publish, imposibleng maitago ang pambu-bulagta.

Subukan mo mismo: baguhin ang `tool_name` o `agent_id` o `timestamp` sa cell sa itaas at patakbuhin muli. Bawat pagbabago ay nagreresulta sa isang hindi valid na resibo.


## Seksyon 3: Ipagsunod-sunod ang mga resibo

Isang resibo ang nagpoprotekta sa isang aksyon. Karamihan sa mga ahente ay gumagawa ng maraming aksyon. Upang gawing ebidensya ng pagbabago ang buong pagkakasunod-sunod, kinakabit namin ang bawat resibo sa naunang isa sa pamamagitan ng pagsama ng hash ng naunang resibo sa payload ng bagong resibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Kung may aalis o magpapalit ng ayos ng resibo, sa eksaktong puntong iyon madudurog ang kadena. Babagsak ang beripikasyon ng anumang mas huling resibo dahil ang `previous_receipt_hash` nito ay hindi na tumutugma sa aktwal na hash ng nauna rito.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Ngayon putulin ang kadena sa pamamagitan ng pag-tamper sa gitnang resibo at muling pag-verify. Nabibigo ang resibong na-tamper sa kanyang tseke ng pirma, AT ang susunod na resibo ay nabibigo sa tseke ng chain-link nito (dahil ang `previous_receipt_hash` nito ay hindi na tumutugma sa hash ng nabagong gitnang resibo).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Nakumpirma pa rin ang Resibo 0 (hindi ito binago at walang naunang resibong naka-depende). Nabigo ang Resibo 1 sa pagsusuri ng pirma dahil binago namin ang `tool_args_hash`. Nabigo ang Resibo 2 sa pagsusuri ng chain-link dahil ang `previous_receipt_hash` nito ay kinompyut laban sa orihinal (na ngayon ay binagong) resibo 1.

Kahit na muling pirmahan ng isang umaatake ang binagong resibo 1 (na hindi nila magagawa nang wala ang pribadong susi), ang hindi tugmang chain-link sa resibo 2 ay magpapakita pa rin ng panlilinlang. Upang maitago ang pagbabago, kailangang muling pirmahan ng umaatake ang bawat resibo mula sa punto ng pagbabago pataas, na nangangailangan ng pag-aari ng pribadong susi.


## Seksyon 4: Balutin ang pagtawag ng tool ng ahente gamit ang pagpirma ng resibo

Sa aktwal na deployment, ayaw mong bawat may-akda ng ahente ay alalahanin na tawagin ang `make_receipt`. Gusto mo na ang pagpirma ng resibo ay awtomatiko para sa bawat pagtawag ng tool.

Narito ang pinakasimpleng pattern: isang wrapper na klase na kumukuha ng anumang callable na function ng tool at nagbabalik ng bersyon nito na naglalabas ng resibo. Ito ay umaangkop sa anumang framework ng ahente, kabilang ang Microsoft Agent Framework (`agent_framework.foundry`).

Kung wala kang naka-set up na project ng Microsoft Foundry, ipinapakita pa rin ng lokal na mock sa ibaba ang pattern.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Pagsasama sa Microsoft Agent Framework

Ang `ReceiptedTool` wrapper sa itaas ay hindi nakadepende sa framework. Para gamitin ito sa loob ng isang agent na ginawa gamit ang Microsoft Agent Framework, irehistro ang naka-wrap na function bilang isang tool. Isang sketch (papalitan mo ang mock ng totoong Microsoft Foundry tool registration):

```python
# Pseudocode na nagpapakita ng anyo ng integrasyon.
# import os
# mula sa agent_framework.foundry import FoundryChatClient
# mula sa azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Ikaw ay isang Contoso Travel agent ...",
#     tools=[receipted_lookup],   # ang naka-wrap na tool, hindi ang raw function
# )
# response = agent.run("Hanapin ang mga flight mula Sydney papuntang Los Angeles sa Hunyo.")
#
# # Pagkatapos ng run, bawat pagtawag ng tool na ginawa ng agent ay may pirmahang resibo:
# audit_chain = receipted_lookup.receipts
```

Hindi kailangang malaman ng agent framework ang anumang tungkol sa mga resibo. Ang paglagda sa resibo ay nakabalot sa tool, hindi ipinaloob sa framework. Ganito ka nagdaragdag ng pinagmulan sa umiiral na agent code nang hindi nire-rewrite ang agent.


## Balik-aral at hamon sa pag-extend

Nagawa mo:

- Nakabuo ng isang Ed25519 na pares ng susi.
- Nakagawa at napirmahan ang isang resibo para sa tawag ng tool ng ahente.
- Na-verify ang resibo offline gamit lamang ang pampublikong susi.
- Na-manipulate ang isang resibo at napansin ang pagbagsak sa pag-verify.
- Nakagawa ng hash-chained na sunod-sunod na tatlong resibo.
- Na-manipulate ang gitna ng chain at napansin ang parehong pagkabigo sa pirma at pagkaputol ng chain-link.
- Naka-wrap ang isang tool function gamit ang awtomatikong paglagda ng resibo.

**Hamon sa pag-extend.** Palawakin ang eskema ng resibo gamit ang `request_id` na field (isang UUID para sa distributed tracing). I-update ang `make_receipt` upang isama ito, at tiyaking ang mga resibo ay nagpapatotoo pa rin nang buo. Pagkatapos, baguhin ang field matapos ang paglagda at tiyaking pumalya ang pag-verify. Pinipilit ka nitong intindihin kung paano bawat byte ng canonical encoding ay nakakatulong sa pirma.

**Mahalagang hangganan.** Pinatutunayan ng mga resibo ang tatlong bagay at tatlong bagay lamang: attribution (ang susi na ito ang pumirma sa nilalaman), integridad (hindi nagbago ang nilalaman mula nang pirmahan), at pagkakasunod-sunod (ang resibong ito ay sumunod sa resibong iyan). Hindi nito pinatutunayan na tama ang aksyon ng ahente, na ang polisiya na may pangalang `policy_id` ay talagang na-evaluate, o na sinunod ng ahente ang bawat patakaran. Ang mga resibo ay pundasyon. Ang pamamahala ang sistemang itinatayo mo sa ibabaw nito.

Basahin muli ang lesson README na ito habang isinasaisip ang hangganang iyon. Ang pinaka-karaniwang pagkakamali ng mga koponan tungkol sa mga resibo ay ang pag-aakala na "may resibo tayo" ay nangangahulugang "may pamamahala tayo." Hindi ito totoo. Ginagawa ng mga resibo na masusuri ang kilos ng ahente. Hindi nito ginagawang tama ito.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
